# FreeSurfer
### adding all the freesurfer tables together and adding the healthy diseased column

In [1]:
import os
import pandas as pd
import glob
import re

# Define the input and output folders
input_folder = r"D:\image_group_data\team44\output_for_freesurfer_table"
output_folder = r"D:\image_group_data\team44\output_for_freesurfer_table\output"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Initialize an empty dictionary to hold subject data
data_dict = {}

# Function to extract subject data from each file
def extract_subject_data(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
        header = lines[0].strip().split()  # First line is the header
        values = lines[1].strip().split()  # Second line contains the values
        return dict(zip(header, values))

# Define regex patterns for healthy and diseased files
healthy_pattern = re.compile(r"subH\d+.*\.txt|h\d+sub.*\.txt|team44HC4sub\d+.*\.txt|team44HC1sub\d+.*\.txt")
diseased_pattern = re.compile(r"sub(?!H)\d+.*\.txt|team44OLF1sub\d+.*\.txt")

# Loop through all files in the input folder
for file in glob.glob(os.path.join(input_folder, "*.txt")):
    subject_id = os.path.basename(file).split("_")[0]  # Extract subject ID (e.g., sub16 or h1sub13)
    if subject_id not in data_dict:
        data_dict[subject_id] = {}
    # Extract data and update the dictionary
    data_dict[subject_id].update(extract_subject_data(file))
    # Assign label based on the pattern
    if healthy_pattern.match(os.path.basename(file)):
        data_dict[subject_id]['Label'] = 0  # Healthy
    elif diseased_pattern.match(os.path.basename(file)):
        data_dict[subject_id]['Label'] = 1  # Diseased

# Convert the dictionary to a DataFrame
df = pd.DataFrame.from_dict(data_dict, orient='index')

# List of columns to drop
columns_to_remove = [
    "lh.aparc.thickness", "lh.aparc.area", "lh.aparc.volume",
    "lh.aparc.meancurv", "lh.aparc.gauscurv", "lh.aparc.foldind",
    "lh.aparc.curvind", "rh.aparc.thickness", "rh.aparc.area",
    "rh.aparc.volume", "rh.aparc.meancurv", "rh.aparc.gauscurv",
    "rh.aparc.foldind", "rh.aparc.curvind", "Measure:volume"
]

# Drop the specified columns if they exist in the DataFrame
df = df.drop(columns=[col for col in columns_to_remove if col in df.columns], errors='ignore')


# Specify the output file path
output_file = os.path.join(output_folder, "compiled_mri_data_cleaned.csv")

# Delete the existing file if it exists
if os.path.exists(output_file):
    os.remove(output_file)

# Save the compiled table to the output folder
df.to_csv(output_file, index=True)

print(f"Table successfully created! File saved at: {output_file}")


Table successfully created! File saved at: D:\image_group_data\team44\output_for_freesurfer_table\output\compiled_mri_data_cleaned.csv


## cleaning the data (FreeSurfer)

Step 1: Handle missing values

Step 2: Remove columns with the same value in all rows

Step 3: Outlier Detection and Replacement

Step 4: Normalize the data

Step 5: Correlation Analysis

Step 6: Feature Engineering (off)

##### Step 7: Remove Specific Subjects Completely
##### Step 8: Remove Specific Hemispheres for Specific Subjects


In [2]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind

# Define input and output folders
input_folder = r"D:\image_group_data\team44\output_for_freesurfer_table\output"
output_folder = r"D:\image_group_data\team44\output_for_freesurfer_table\output"
input_file = os.path.join(input_folder, "compiled_mri_data_cleaned.csv")
output_file_all = os.path.join(output_folder, "cleaned_mri_data.csv")
output_file_subjects_removed = os.path.join(output_folder, "cleaned_mri_data_subjects_removed.csv")
output_file_hemisphere_removed = os.path.join(output_folder, "cleaned_mri_data_hemisphere_removed.csv")

# Load your data
df = pd.read_csv(input_file)

# Ensure renaming of the 'Unnamed: 0' column to 'Subject_ID'
if "Unnamed: 0" in df.columns:
    df.rename(columns={"Unnamed: 0": "Subject_ID"}, inplace=True)

# Ensure 'Subject_ID' exists
if "Subject_ID" not in df.columns:
    raise KeyError("'Subject_ID' column is missing from the dataset. Please check the input file.")

# Switches for each step
perform_missing_value_imputation = True
perform_constant_column_removal = True
perform_outlier_detection = True
perform_normalization = False
perform_correlation_analysis = False  # Turn off correlation analysis to avoid redundancy
perform_ttest_analysis = True         # New switch for t-test feature selection
perform_feature_engineering = False
remove_specific_subjects = True
remove_specific_hemispheres = True

# Step 1: Handle missing values  ==============================================================
if perform_missing_value_imputation:
    def impute_missing_values(df, method='median'):
        numeric_cols = df.select_dtypes(include=['number']).columns
        if method == 'median':
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        elif method == 'mean':
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
        return df

    # Separate groups and create copies
    patients = df[df['Label'] == 1].copy()
    controls = df[df['Label'] == 0].copy()

    # Impute missing values only for numeric columns
    patients = impute_missing_values(patients, method='median')
    controls = impute_missing_values(controls, method='mean')

    # Combine the groups back
    df_cleaned = pd.concat([patients, controls], axis=0)
else:
    df_cleaned = df.copy()

# Step 2: Remove columns with the same value in all rows======================================
if perform_constant_column_removal:
    constant_columns = [col for col in df_cleaned.columns if df_cleaned[col].nunique() == 1]
    df_cleaned = df_cleaned.drop(columns=constant_columns, errors='ignore')
    print(f"Removed constant columns: {constant_columns}")

# Step 3: Outlier Detection and Replacement===================================================
if perform_outlier_detection:
    def detect_and_replace_outliers_by_group(df, group_column, threshold=3, replacement_method='mean'):
        df_cleaned = df.copy()
        numeric_cols = df.select_dtypes(include=['number']).columns
        outlier_details = []

        for group in df[group_column].unique():
            group_data = df[df[group_column] == group]
            means = group_data[numeric_cols].mean()
            stds = group_data[numeric_cols].std()

            stds_replaced = stds.replace(0, 1)
            z_scores = (group_data[numeric_cols] - means) / stds_replaced
            outliers = (z_scores.abs() > threshold)

            for row_idx, row in outliers.iterrows():
                for col in numeric_cols:
                    if row[col]:
                        subject_id = df.at[row_idx, 'Subject_ID']
                        outlier_details.append((subject_id, col, group))
                        print(f"Outlier found: Subject '{subject_id}', Column '{col}', Group '{group}'")

                        replacement_value = means[col] if replacement_method == 'mean' else group_data[col].median()
                        df_cleaned.at[row_idx, col] = replacement_value

        return df_cleaned, outlier_details

    replacement_method = 'mean'
    df_cleaned, outlier_details = detect_and_replace_outliers_by_group(
        df_cleaned, group_column='Label', threshold=5, replacement_method=replacement_method
    )
    print(f"Outliers replaced using '{replacement_method}' method.")

# Step 4: Normalize the data=============================================================================
if perform_normalization:
    scaler = StandardScaler()
    features_to_normalize = [col for col in df_cleaned.columns if col not in ['Label', 'Subject_ID']]
    df_cleaned[features_to_normalize] = scaler.fit_transform(df_cleaned[features_to_normalize])

# Step 5: Correlation Analysis ==========================================================================
if perform_correlation_analysis:
    correlation_threshold = 0.95  # Define threshold for removing correlated features

    # Select only numeric columns for correlation
    numeric_cols = df_cleaned.select_dtypes(include=['number']).columns
    correlation_matrix = df_cleaned[numeric_cols].corr()

    high_correlation_columns = set()

    for col in correlation_matrix.columns:
        for row in correlation_matrix.index:
            if col != row and abs(correlation_matrix.loc[row, col]) > correlation_threshold:
                if row not in high_correlation_columns:
                    high_correlation_columns.add(row)

    # Drop highly correlated features
    df_cleaned = df_cleaned.drop(columns=high_correlation_columns, errors='ignore')
    print(f"Removed highly correlated columns: {high_correlation_columns}")
    
# Step 5: Ttest Analysis ==========================================================================
if perform_ttest_analysis:
    def t_test_feature_selection(df, label_col='Label', alpha=0.05):
        """
        Performs a t-test between patients (label=1) and controls (label=0) for each numeric feature.
        Keeps features whose p-value < alpha and removes those with p-value >= alpha.
        """
        # Separate patients and controls
        patients = df[df[label_col] == 1]
        controls = df[df[label_col] == 0]

        # Identify numeric columns (excluding 'Label' and 'Subject_ID')
        numeric_cols = df.select_dtypes(include=['number']).columns
        numeric_cols = [col for col in numeric_cols if col not in [label_col, 'Subject_ID']]

        # Check if there are any numeric columns to analyze
        if not numeric_cols:
            print("No numeric features found for t-test analysis.")
            return df  # Return the original dataframe if no numeric columns

        removed_features = []
        kept_features = []

        # Perform t-test for each numeric column
        for col in numeric_cols:
            t_stat, p_value = ttest_ind(
                patients[col], 
                controls[col], 
                equal_var=False,  # Welch's t-test
                nan_policy='omit'  # Handle NaN values
            )
            if p_value < alpha:
                kept_features.append(col)
            else:
                removed_features.append(col)

        # Drop the removed features
        df = df.drop(columns=removed_features, errors='ignore')

        # Print the summary of results
        print(f"Total numeric features before T-test: {len(numeric_cols)}")
        print(f"Removed features (p >= {alpha}): {len(removed_features)}")
        print(f"Kept features (p < {alpha}): {len(kept_features)}")
        print(f"Total features left after T-test: {len(kept_features)}")

        return df

    # Apply t-test feature selection
    df_cleaned = t_test_feature_selection(df_cleaned, label_col='Label', alpha=0.05)

# Step 6: Feature Engineering ==========================================================================
if perform_feature_engineering:
    def create_features(df):
        # Example transformations
        if 'Age' in df.columns and 'Gender' in df.columns:
            df['Age_Gender_Interaction'] = df['Age'] * (df['Gender'] == 'Male').astype(int)

        # Example normalization
        if 'BrainVolume' in df.columns and 'CortexVolume' in df.columns:
            df['Cortex_Brain_Ratio'] = df['CortexVolume'] / df['BrainVolume']

        return df

    df_cleaned = create_features(df_cleaned)
    print("Feature engineering completed.")

# Save cleaned data
os.makedirs(output_folder, exist_ok=True)
df_cleaned.to_csv(output_file_all, index=False)
print(f"Data cleaned and saved at: {output_file_all}")

################################################################################################################
# Step 7: Remove Specific Subjects Completely
#---------------------------------------------------------------------------------------------------------------
if remove_specific_subjects:
    subjects_to_remove = ["sub20", "sub34","sub39","sub21","sub16", "sub22","sub30","sub31","sub41",
                          'team44HC4sub57','team44OLF1sub01','team44OLF1sub02','team44OLF1sub03','team44OLF1sub04',
                          'team44OLF1sub06','team44OLF1sub13','team44OLF1sub15']  # Add Subject_IDs to be removed
    df_subjects_removed = df_cleaned[~df_cleaned['Subject_ID'].isin(subjects_to_remove)].copy()
    df_subjects_removed.to_csv(output_file_subjects_removed, index=False)
    print(f"Data with specified subjects removed saved at: {output_file_subjects_removed}")

# Step 8: Remove Specific Hemispheres for Specific Subjects
#---------------------------------------------------------------------------------------------------------------
if remove_specific_hemispheres:
    remove_subjects_hemispheres = {
        "sub16": ["lh"],
        "sub20": ["lh", "rh"],
        "sub21": [ "rh"],
        "sub22": [ "rh"],
        "sub30": ["rh"],
        "sub31": ["lh"],
        "sub34": ["lh", "rh"],
        "sub35": ["lh", "rh"],
        "sub39": ["lh", "rh"],
        "sub41": ["lh"],
        'team44HC4sub57': ["lh", "rh"],
        'team44OLF1sub01': ["lh"],
        'team44OLF1sub02': ["lh", "rh"],
        'team44OLF1sub03': ["lh", "rh"],
        'team44OLF1sub04': ["lh"],
        'team44OLF1sub06': ["lh", "rh"],
        'team44OLF1sub15': ["lh", "rh"], 
    }
    
    def remove_subject_hemisphere_data(df, subjects_hemispheres):
        df_modified = df.copy()
        hemisphere_keywords = {"lh": "Left", "rh": "Right"}

        # Identify ambiguous columns
        ambiguous_columns = [
            col for col in df.columns
            if not any(keyword in col.lower() for keyword in ["lh", "rh", "left", "right"]) and col not in ["Subject_ID", "Label"]
        ]

        print(f"Removing ambiguous columns: {ambiguous_columns}")
        df_modified = df_modified.drop(columns=ambiguous_columns, errors='ignore')

        for subject, hemispheres in subjects_hemispheres.items():
            if subject not in df['Subject_ID'].values:
                print(f"Warning: Subject '{subject}' not found in dataset.")
                continue
    
            if not isinstance(hemispheres, list):  # Ensure hemispheres is a list
                hemispheres = [hemispheres]

            for hemisphere in hemispheres:
                prefix = hemisphere_keywords.get(hemisphere, "")
                hemisphere_columns = [
                    col for col in df.columns
                    if (col.startswith(prefix) or hemisphere in col.lower()) and col not in ["Subject_ID", "Label"]
                ]

                for col in hemisphere_columns:
                    if col in df_modified.columns:
                        df_modified.loc[df_modified['Subject_ID'] == subject, col] = None

                print(f"Processed hemisphere '{hemisphere}' for subject '{subject}'.")

        return df_modified


    df_hemisphere_removed = remove_subject_hemisphere_data(df_cleaned.copy(), remove_subjects_hemispheres)
    df_hemisphere_removed.to_csv(output_file_hemisphere_removed, index=False)
    print(f"Data with specified subjects and their hemisphere data removed saved at: {output_file_hemisphere_removed}")


Removed constant columns: ['Left-WM-hypointensities', 'Right-WM-hypointensities', 'Left-non-WM-hypointensities', 'Right-non-WM-hypointensities']
Outlier found: Subject 'team44OLF1sub06', Column 'rh_entorhinal_foldind', Group '1'
Outlier found: Subject 'team44OLF1sub06', Column 'WM-hypointensities', Group '1'
Outlier found: Subject 'team44OLF1sub14', Column '5th-Ventricle', Group '1'
Outlier found: Subject 'subH06', Column 'rh_insula_foldind', Group '0'
Outlier found: Subject 'h2sub25', Column '5th-Ventricle', Group '0'
Outlier found: Subject 'h3sub36', Column 'lh_frontalpole_foldind', Group '0'
Outlier found: Subject 'h3sub41', Column 'rh_parahippocampal_volume', Group '0'
Outlier found: Subject 'h4sub51', Column 'lh_precuneus_gauscurv', Group '0'
Outlier found: Subject 'team44HC4sub64', Column 'non-WM-hypointensities', Group '0'
Outliers replaced using 'mean' method.
Total numeric features before T-test: 541
Removed features (p >= 0.05): 337
Kept features (p < 0.05): 204
Total feature

### seperating right hemesphear from left hemesphear data 

In [3]:
import pandas as pd

# Load the hemisphere data
input_file_hemisphere_removed = r"D:\image_group_data\team44\output_for_freesurfer_table\output\cleaned_mri_data_hemisphere_removed.csv"
output_file_left_hemisphere = r"D:\image_group_data\team44\output_for_freesurfer_table\output\left_hemisphere_mri_data_filtered.csv"
output_file_right_hemisphere = r"D:\image_group_data\team44\output_for_freesurfer_table\output\right_hemisphere_mri_data_filtered.csv"

# Load the data
df = pd.read_csv(input_file_hemisphere_removed)

# Separate columns into Left (lh) and Right (rh)
left_columns = [col for col in df.columns if col.lower().startswith(('lh', 'left'))]
right_columns = [col for col in df.columns if col.lower().startswith(('rh', 'right'))]

# Create Left Hemisphere DataFrame
left_hemisphere_df = df[['Subject_ID', 'Label'] + left_columns].copy()

# Drop rows with any empty cells for the left hemisphere dataset
left_hemisphere_df = left_hemisphere_df.dropna()

# Create Right Hemisphere DataFrame
right_hemisphere_df = df[['Subject_ID', 'Label'] + right_columns].copy()

# Drop rows with any empty cells for the right hemisphere dataset
right_hemisphere_df = right_hemisphere_df.dropna()

# Save the filtered data
left_hemisphere_df.to_csv(output_file_left_hemisphere, index=False)
right_hemisphere_df.to_csv(output_file_right_hemisphere, index=False)

print(f"Filtered left hemisphere data saved to: {output_file_left_hemisphere}")
print(f"Filtered right hemisphere data saved to: {output_file_right_hemisphere}")


Filtered left hemisphere data saved to: D:\image_group_data\team44\output_for_freesurfer_table\output\left_hemisphere_mri_data_filtered.csv
Filtered right hemisphere data saved to: D:\image_group_data\team44\output_for_freesurfer_table\output\right_hemisphere_mri_data_filtered.csv


# conn graph theory feature extraction

In [4]:
import os
import numpy as np
import scipy.io
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import community.community_louvain as community
from scipy.stats import entropy

# Helper Functions
def calculate_network_entropy(degree_distribution):
    degree_sum = np.sum(degree_distribution)
    if degree_sum == 0:
        return 0.0
    probabilities = degree_distribution / degree_sum
    return entropy(probabilities)

def calculate_flow_hierarchy(G):
    directed_graph = nx.DiGraph(G)
    return nx.flow_hierarchy(directed_graph)

def calculate_spectral_entropy(laplacian_spectrum):
    abs_spectrum = np.abs(laplacian_spectrum)
    total = np.sum(abs_spectrum)
    if total == 0:
        return 0.0
    probs = abs_spectrum / total
    probs = np.where(probs > 0, probs, 1e-10)
    return -np.sum(probs * np.log(probs))

def calculate_burt_constraint(G):
    return np.mean(list(nx.constraint(G).values()))

def calculate_robustness(G):
    largest_cc = max(nx.connected_components(G), key=len)
    size_before = len(largest_cc)
    G_copy = G.copy()
    G_copy.remove_edges_from(list(G_copy.edges())[:len(G_copy.edges()) // 2])
    largest_cc_after = max(nx.connected_components(G_copy), key=len)
    size_after = len(largest_cc_after)
    return size_after / size_before

def calculate_graphlet_counts(G):
    """
    Count occurrences of specific small subgraph patterns (graphlets).
    """
    triangles = sum(nx.triangles(G).values()) / 3
    squares = sum(1 for cycle in nx.cycle_basis(G) if len(cycle) == 4)
    return triangles, squares

def calculate_community_conductance(G, partition):
    """
    Calculate conductance for all communities in the graph.
    """
    total_conductance = 0
    for community_id in set(partition.values()):
        nodes_in_community = [n for n, c in partition.items() if c == community_id]
        edges_inside = G.subgraph(nodes_in_community).number_of_edges()
        edges_outside = sum(dict(G.degree(nodes_in_community)).values())
        conductance = edges_outside / max(edges_inside + edges_outside, 1e-10)
        total_conductance += conductance
    return total_conductance / len(set(partition.values()))


def calculate_community_expansion(G, partition):
    """
    Calculate expansion (edge-to-node ratio) for all communities in the graph.
    """
    total_expansion = 0
    for community_id in set(partition.values()):
        nodes_in_community = [n for n, c in partition.items() if c == community_id]
        edges_outside = sum(dict(G.degree(nodes_in_community)).values())
        expansion = edges_outside / max(len(nodes_in_community), 1e-10)
        total_expansion += expansion
    return total_expansion / len(set(partition.values()))


# Main Processing Function
def process_single_file(file_path, health_label):
    mat_data = scipy.io.loadmat(file_path)
    Z = mat_data['Z']
    df = pd.DataFrame(Z)
    np.fill_diagonal(df.values, 0)

    # Normalize the matrix
    df_abs = df.abs()
    scaler = MinMaxScaler()
    df_normalized = pd.DataFrame(scaler.fit_transform(df_abs), columns=df.columns, index=df.index)

    # Apply thresholding
    max_value = df_normalized.where(~np.eye(len(df_normalized), dtype=bool)).max().max()
    min_value = df_normalized.where(~np.eye(len(df_normalized), dtype=bool)).min().min()
    threshold = min_value + 0.65 * (max_value - min_value)
    adjacency_matrix = (df_normalized > threshold).astype(int).values

    # Create graph
    G = nx.Graph(adjacency_matrix)

    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        print(f"Graph {file_path} is empty or sparse. Skipping computation.")
        return {
            'File': os.path.splitext(os.path.basename(file_path))[0],
            'EdgeCount': 0,
            'NodeCount': 0,
            'HealthStatus': health_label
        }

    # If the graph is disconnected, keep only the largest connected component
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    # Partition the graph into communities
    partition = community.best_partition(G)

    # Compute centralities
    centrality_metrics = {
        'BetweennessCentrality': nx.betweenness_centrality(G),
        'ClosenessCentrality': nx.closeness_centrality(G),
        'SubgraphCentrality': nx.subgraph_centrality(G),
        'KatzCentrality': nx.katz_centrality_numpy(G),
        'EigenvectorCentrality': nx.eigenvector_centrality_numpy(G),
        'HarmonicCentrality': nx.harmonic_centrality(G),
        'LoadCentrality': nx.load_centrality(G)
    }

    # Graphlet counts
    triangles, squares = calculate_graphlet_counts(G)

    # Community metrics
    community_conductance = calculate_community_conductance(G, partition)
    community_expansion = calculate_community_expansion(G, partition)

    # Spectral features
    laplacian_spectrum = nx.laplacian_spectrum(G)
    spectral_radius = max(laplacian_spectrum) if len(laplacian_spectrum) > 0 else 0.0
    spectral_entropy = calculate_spectral_entropy(laplacian_spectrum)
    graph_energy = np.sum(np.abs(nx.adjacency_spectrum(G)))

    # Community entropy
    community_sizes = list(pd.Series(list(partition.values())).value_counts())
    community_entropy = entropy(community_sizes)

    # Additional metrics
    k_connectivity = nx.node_connectivity(G)
    edge_fragility = nx.edge_connectivity(G)
    critical_threshold = spectral_radius
    number_of_cliques = sum(1 for _ in nx.find_cliques(G))
    degree_sequence = np.array([d for n, d in G.degree()])
    network_entropy = calculate_network_entropy(degree_sequence)

    # Diameter and average commute time
    diameter = nx.diameter(G) if nx.is_connected(G) else None
    avg_commute_time = nx.average_shortest_path_length(G) if nx.is_connected(G) else None

    # Flow hierarchy
    flow_hierarchy = calculate_flow_hierarchy(G) if nx.is_connected(G) else 0.0

    # Additional features from helper function
    # Merge additional metrics



    # Prepare row_data
    row_data = {
        'File': os.path.splitext(os.path.basename(file_path))[0],
        'EdgeCount': G.number_of_edges(),
        'NodeCount': G.number_of_nodes(),
        'AverageDegree': np.mean(degree_sequence) if len(degree_sequence) > 0 else 0.0,
        'GraphDensity': nx.density(G),
        'GraphRadius': nx.radius(G) if nx.is_connected(G) else None,
        'Diameter': diameter,
        'EdgeConnectivity': edge_fragility,
        'Modularity': community.modularity(partition, G),
        'CommunityEntropy': community_entropy,
        'CommunityConductance': community_conductance,
        'CommunityExpansion': community_expansion,
        'AverageClusteringCoefficient': nx.average_clustering(G),
        'GlobalEfficiency': nx.global_efficiency(G),
        'CharacteristicPathLength': avg_commute_time,
        'FlowHierarchy': flow_hierarchy,
        'AlgebraicConnectivity': nx.algebraic_connectivity(G) if nx.is_connected(G) else None,
        'Assortativity': nx.degree_assortativity_coefficient(G),
        'Transitivity': nx.transitivity(G),
        'LocalEfficiency': nx.local_efficiency(G),
        'BetweennessCentrality': np.mean(list(centrality_metrics['BetweennessCentrality'].values())),
        'ClosenessCentrality': np.mean(list(centrality_metrics['ClosenessCentrality'].values())),
        'SubgraphCentrality': np.mean(list(centrality_metrics['SubgraphCentrality'].values())),
        'KatzCentrality': np.mean(list(centrality_metrics['KatzCentrality'].values())),
        'EigenvectorCentrality': np.mean(list(centrality_metrics['EigenvectorCentrality'].values())),
        'HarmonicCentrality': np.mean(list(centrality_metrics['HarmonicCentrality'].values())),
        'LoadCentrality': np.mean(list(centrality_metrics['LoadCentrality'].values())),
        'TriangleCount': triangles,
        'SquareCount': squares,
        'SpectralRadius': spectral_radius,
        'SpectralEntropy': spectral_entropy,
        'GraphEnergy': graph_energy,
        'NetworkEntropy': network_entropy,
        'KConnectivity': k_connectivity,
        'CriticalThreshold': critical_threshold,
        'NumberOfCliques': number_of_cliques,
        'GraphRobustness': calculate_robustness(G),
        'HealthStatus': health_label
    }
    additional_metrics = calculate_additional_features(G, partition)
    for key, value in additional_metrics.items():
        row_data[key] = value if value is not None else 'N/A'  # Replace None with 'N/A'

    # Merge additional metrics
    row_data.update(additional_metrics)

    return row_data



def calculate_additional_features(G, partition):
    """
    Compute additional advanced graph features.
    """
    # Higher-order graphlets
    wedges = sum(1 for node in G if G.degree[node] == 2)  # Approximation for wedges
    k_cycles = sum(1 for cycle in nx.cycle_basis(G) if len(cycle) > 4)  # Cycles > 4 nodes

    # Structural holes
    efficiency = np.mean([nx.efficiency(G, u, v) for u, v in nx.non_edges(G)])
    constraint_dist = list(nx.constraint(G).values())
    constraint_variance = np.var(constraint_dist)

    # Articulation points and bridges
    articulation_points = len(list(nx.articulation_points(G)))
    bridges = len(list(nx.bridges(G)))

    # Path-based metrics
    # Skip max_flow and min_cut for undirected graphs
    max_flow = None  # Not applicable for undirected graphs
    min_cut = None   # Not applicable for undirected graphs

    # Community metrics
    intra_community_edges = sum(
        1 for u, v in G.edges if partition[u] == partition[v]
    )
    inter_community_edges = G.number_of_edges() - intra_community_edges
    intra_community_density = intra_community_edges / max(1, len(partition))
    inter_community_fraction = inter_community_edges / max(1, G.number_of_edges())

    # Spectral metrics
    laplacian_spectrum = nx.laplacian_spectrum(G)
    spectral_gap = (laplacian_spectrum[0] - laplacian_spectrum[1]) if len(laplacian_spectrum) > 1 else 0
    effective_resistance = sum(1.0 / eig for eig in laplacian_spectrum if eig > 0)
    laplacian_energy = sum(eig**2 for eig in laplacian_spectrum)

    return {
        'WedgeCount': wedges,
        'KCycleCount': k_cycles,
        'Efficiency': efficiency,
        'ConstraintVariance': constraint_variance,
        'ArticulationPoints': articulation_points,
        'Bridges': bridges,
        'MaximumFlow': max_flow,  # Return None
        'MinimumCut': min_cut,    # Return None
        'IntraCommunityDensity': intra_community_density,
        'InterCommunityFraction': inter_community_fraction,
        'SpectralGap': spectral_gap,
        'EffectiveResistance': effective_resistance,
        'LaplacianEnergy': laplacian_energy,
    }


def process_directory(directory_path, health_label):
    rows = []
    for filename in os.listdir(directory_path):
        if filename.endswith('.mat'):
            file_path = os.path.join(directory_path, filename)
            row_data = process_single_file(file_path, health_label)
            print(f"{row_data['File']} has been processed! HealthStatus = {health_label}")
            rows.append(row_data)
    return rows
# The rest of your code remains unchanged.
# Example setup for three groups. You can rename the CSV files per group as you wish.
output_folder = r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output'

patient_dirs = [
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_all_no_put_olfactory',            # Group 1 Patients
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_putamen_olfactory',     # Group 2 Patients
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_table_olfactory',      # Group 3 Patients
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\whole_brain\healthy\whole_brain',
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\whole_brain\healthy\whole_brain_conn'
]

healthy_dirs = [
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_all_no_put_healthy',         # Group 1 Healthy
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_putamen_healthy', # Group 2 Healthy
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\RRC_table_only_healthy',  # Group 3 Healthy
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\whole_brain\olfactory\whole_brain',
    r'D:\image_group_data\team44\CONN_Team44\all_first_levlels\whole_brain\olfactory\whole_brain_conn'
]
# Define custom output names
output_filenames = [
    'all_no_put.csv',
    'putamen.csv',
    'table_only.csv',
    'whole_brain.csv',
    'whole_brain_conn.csv'
]

for p_dir, h_dir, out_name in zip(patient_dirs, healthy_dirs, output_filenames):
    patient_data = process_directory(p_dir, health_label=1)
    healthy_data = process_directory(h_dir, health_label=0)
    result_df = pd.DataFrame(patient_data + healthy_data)

    output_file_path = os.path.join(output_folder, out_name)
    result_df.to_csv(output_file_path, index=False)
    print(f"CSV file saved as {output_file_path}")


resultsROI_Subject001_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject002_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject003_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject004_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject005_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject006_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject007_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject008_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject009_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject010_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject011_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject012_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject013_Condition001 has been processed! HealthStatus = 1
resultsROI_Subject014_Condition001 has been processed! HealthSta

 ## cleaning CONN datasets 
Step 1: Handle missing values

Step 2: Remove constant columns

Step 3: Outlier detection and replacement
    
Step 4: Normalize the data

Step 5: Correlation analysis | 
Step 5: Ttest analysis

Step 6: Feature engineering (off)

Step 7: Save cleaned dataset

 Step 8: Remove specific subjects

In [5]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind


# Define input and output paths for datasets
datasets = [
    {
        "input_file": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\all_no_put.csv",
        "output_cleaned": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\all_no_put_cleaned.csv",
        "output_subjects_removed": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\all_no_put_subjects_removed.csv",
    },
    {
        "input_file": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\putamen.csv",
        "output_cleaned": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\putamen_cleaned.csv",
        "output_subjects_removed": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\putamen_subjects_removed.csv",
    },
    {
        "input_file": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\table_only.csv",
        "output_cleaned": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\table_only_cleaned.csv",
        "output_subjects_removed": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\table_only_subjects_removed.csv",
    },
    {
        "input_file": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain.csv",
        "output_cleaned": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_cleaned.csv",
        "output_subjects_removed": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_subjects_removed.csv",
    },
    {
        "input_file": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_conn.csv",
        "output_cleaned": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_conn_cleaned.csv",
        "output_subjects_removed": r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_conn_subjects_removed.csv",
    }
]

# Switches for each step
perform_missing_value_imputation = True
perform_constant_column_removal = True
perform_outlier_detection = True
perform_normalization = False
perform_correlation_analysis = False  # Set to False if using t-test instead
perform_ttest_analysis = True         # New switch for t-test
perform_feature_engineering = False
remove_specific_subjects = True


def remove_nan_columns(df):
    """
    Removes columns that are entirely NaN from the DataFrame.
    """
    nan_columns = df.columns[df.isna().all()]
    print(f"Removed columns with all NaN values: {list(nan_columns)}")
    return df.drop(columns=nan_columns, errors='ignore')

# Helper functions
def impute_missing_values(df, method='median'):
    numeric_cols = df.select_dtypes(include=['number']).columns
    if method == 'median':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    elif method == 'mean':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    return df

def detect_and_replace_outliers_by_group(df, group_column, threshold=3, replacement_method='mean'):
    """
    Detects and replaces outliers by group, explicitly casting replacement values to the column's dtype.
    """
    df_cleaned = df.copy()
    numeric_cols = df.select_dtypes(include=['number']).columns

    for group in df[group_column].unique():
        group_data = df[df[group_column] == group]
        means = group_data[numeric_cols].mean()
        stds = group_data[numeric_cols].std()
        stds_replaced = stds.replace(0, 1)
        z_scores = (group_data[numeric_cols] - means) / stds_replaced
        outliers = (z_scores.abs() > threshold)

        for row_idx, row in outliers.iterrows():
            for col in numeric_cols:
                if row[col]:  # If the value is an outlier
                    replacement_value = means[col] if replacement_method == 'mean' else group_data[col].median()
                    # Cast the replacement value to the column's original dtype
                    replacement_value = df_cleaned[col].dtype.type(replacement_value)
                    df_cleaned.at[row_idx, col] = replacement_value

    return df_cleaned

# --------------- T-test Helper Function ---------------
def t_test_feature_selection(df, label_col='Label', alpha=0.05):
    """
    Performs a t-test between patients (label=1) and controls (label=0) for each numeric feature.
    Keeps features whose p-value >= alpha and removes those with p-value < alpha.
    """
    # Separate patients and controls
    patients = df[df[label_col] == 1]
    controls = df[df[label_col] == 0]
    
    # Identify numeric columns (excluding Label and Subject_ID)
    numeric_cols = df.select_dtypes(include=['number']).columns
    numeric_cols = [col for col in numeric_cols if col not in [label_col, 'Subject_ID']]
    
    removed_features = []
    kept_features = []  # For tracking purposes
    
    # Perform t-test for each numeric column
    for col in numeric_cols:
        t_stat, p_value = ttest_ind(
            patients[col], 
            controls[col], 
            equal_var=False, 
            nan_policy='omit'
        )
        
        # Keep features with p-value >= alpha, remove others
        if p_value < alpha:
            removed_features.append(col)
        else:
            kept_features.append(col)
    
    # Drop the removed features
    df = df.drop(columns=removed_features, errors='ignore')
    
    # Print which features were removed and kept
    print(f"Removed features by T-test (p < {alpha}): {removed_features}")
    print(f"Kept features by T-test (p >= {alpha}): {kept_features}")
    print(f"Number of features left after T-test: {df.shape[1]}")
    
    return df


from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import numpy as np

def create_features(df):
    # 1. Centrality Aggregates
    centrality_features = [
        'BetweennessCentrality', 'SubgraphCentrality',
        'KatzCentrality', 'EigenvectorCentrality'
    ]
    df['Centrality_Mean'] = df[centrality_features].mean(axis=1)
    df['Centrality_StdDev'] = df[centrality_features].std(axis=1)

    # 2. Feature Ratios
    if 'Diameter' in df.columns and 'GraphRadius' in df.columns:
        df['Radius_to_Diameter'] = df['GraphRadius'] / (df['Diameter'] + 1e-6)  # Avoid division by zero
    if 'AlgebraicConnectivity' in df.columns and 'CharacteristicPathLength' in df.columns:
        df['Connectivity_to_PathLength'] = df['AlgebraicConnectivity'] / (df['CharacteristicPathLength'] + 1e-6)

    # 3. Polynomial Features (Interaction Terms)
    for f1 in centrality_features:
        for f2 in centrality_features:
            if f1 != f2:
                interaction_name = f'{f1}_x_{f2}'
                df[interaction_name] = df[f1] * df[f2]

    # 4. Clustering Features
    # Only use existing numerical columns excluding 'Label' and 'Subject_ID' if present
    exclude_columns = ['Label', 'Subject_ID']
    numeric_features = df.select_dtypes(include=['number']).drop(
        columns=[col for col in exclude_columns if col in df.columns], errors='ignore'
    )
    if numeric_features.empty:
        raise ValueError("No numeric features available for clustering.")
    
    kmeans = KMeans(n_clusters=3, random_state=42)  # Adjust the number of clusters as needed
    df['Cluster_Label'] = kmeans.fit_predict(numeric_features)

    # 5. Dimensionality Reduction
    pca = PCA(n_components=2, random_state=42)  # Reduce to 2 components for simplicity
    pca_result = pca.fit_transform(numeric_features)
    df['PCA_1'] = pca_result[:, 0]
    df['PCA_2'] = pca_result[:, 1]

    return df


# Main processing loop
for dataset in datasets:
    # Load the data
    df = pd.read_csv(dataset["input_file"])

    # Ensure renaming of the subject ID column
    if "File" in df.columns:  # Change 'File' to 'Subject_ID' if applicable
        df.rename(columns={"File": "Subject_ID"}, inplace=True)
        # Also rename HealthStatus to Label if that's your convention
        df.rename(columns={"HealthStatus": "Label"}, inplace=True)

    # Ensure 'Subject_ID' exists
    if "Subject_ID" not in df.columns:
        raise KeyError("'Subject_ID' column is missing from the dataset. Please check the input file.")
    
    # Step 0: Remove NaN columns ------------------------------------------------------------
    df = remove_nan_columns(df)

    # Step 1: Handle missing values ---------------------------------------------------------
    if perform_missing_value_imputation:
        patients = df[df['Label'] == 1].copy()
        controls = df[df['Label'] == 0].copy()
        patients = impute_missing_values(patients, method='median')
        controls = impute_missing_values(controls, method='mean')
        df_cleaned = pd.concat([patients, controls], axis=0)
    else:
        df_cleaned = df.copy()
    # Step 2: Remove constant columns -------------------------------------------------------
    if perform_constant_column_removal:
        constant_columns = [col for col in df_cleaned.columns if df_cleaned[col].nunique() == 1]
        df_cleaned = df_cleaned.drop(columns=constant_columns, errors='ignore')
        print(f"Removed constant columns: {constant_columns}")

    # Step 3: Outlier detection and replacement --------------------------------------------
    if perform_outlier_detection:
        df_cleaned = detect_and_replace_outliers_by_group(
            df_cleaned, 
            group_column='Label', 
            threshold=5, 
            replacement_method='mean'
        )

    # Step 4: Normalize the data -----------------------------------------------------------
    if perform_normalization:
        scaler = StandardScaler()
        features_to_normalize = [col for col in df_cleaned.columns if col not in ['Label', 'Subject_ID']]
        df_cleaned[features_to_normalize] = scaler.fit_transform(df_cleaned[features_to_normalize])
    # ----------------------------------------------------------------------------
    # Step 5: T-test-based feature selection (REPLACES correlation analysis)
    # ----------------------------------------------------------------------------
    if perform_ttest_analysis:
        # Instead of correlation, we'll do t-test feature selection:
        df_cleaned = t_test_feature_selection(df_cleaned, label_col='Label', alpha=0.05)


    # Step 5: Correlation analysis ------------------------------------------------------------
    if perform_correlation_analysis:
        numeric_cols = df_cleaned.select_dtypes(include=['number']).columns
        correlation_matrix = df_cleaned[numeric_cols].corr()
        high_correlation_columns = set()
        for col in correlation_matrix.columns:
            for row in correlation_matrix.index:
                if col != row and abs(correlation_matrix.loc[row, col]) > 0.95:
                    high_correlation_columns.add(row)
        df_cleaned = df_cleaned.drop(columns=high_correlation_columns, errors='ignore')
        print(f"Removed highly correlated columns: {high_correlation_columns}")

    # Step 6: Feature engineering ------------------------------------------------------------
    if perform_feature_engineering:
        df_cleaned = create_features(df_cleaned)
        print("Feature engineering completed.")

    # Step 7: Save cleaned dataset ------------------------------------------------------------
    df_cleaned.to_csv(dataset["output_cleaned"], index=False)
    print(f"Cleaned data saved to {dataset['output_cleaned']}")

    # Step 8: Remove specific subjects ------------------------------------------------------------
    if remove_specific_subjects:
        # Define subjects to remove as a list of (Subject_ID, Label) tuples
        subjects_to_remove = [("resultsROI_Subject020_Condition001", 1), ("resultsROI_Subject034_Condition001", 1),
                              ("resultsROI_Subject039_Condition001", 1), ("resultsROI_Subject021_Condition001", 1), 
                              ("resultsROI_Subject016_Condition001", 1), ("resultsROI_Subject022_Condition001", 1),
                              ("resultsROI_Subject030_Condition001", 1), ("resultsROI_Subject031_Condition001", 1),
                              ("resultsROI_Subject041_Condition001", 1),
                              ("resultsROI_Subject001_Condition001", 1),("resultsROI_Subject001_Condition002", 1),
                              ("resultsROI_Subject001_Condition003", 1),("resultsROI_Subject001_Condition004", 1),
                              ("resultsROI_Subject001_Condition006", 1),("resultsROI_Subject001_Condition013", 1),
                              ("resultsROI_Subject001_Condition015", 1),("resultsROI_Subject001_Condition057", 0),]
    
        # Create a condition to filter out rows matching both Subject_ID and Label
        condition_to_remove = df_cleaned.apply(lambda row: (row['Subject_ID'], row['Label']) in subjects_to_remove, axis=1)

        # Identify the rows being removed
        removed_rows = df_cleaned[condition_to_remove]
        if not removed_rows.empty:
            print("The following subjects were found and removed:")
            print(removed_rows[['Subject_ID', 'Label']])
        else:
            print("No specified subjects were found in the dataset.")

        # Remove the rows
        df_subjects_removed = df_cleaned[~condition_to_remove].copy()

        # Save the cleaned dataset with specific subjects removed
        df_subjects_removed.to_csv(dataset["output_subjects_removed"], index=False)
        print(f"Data with specified subjects removed saved to {dataset['output_subjects_removed']}")



Removed columns with all NaN values: ['MaximumFlow', 'MinimumCut']
Removed constant columns: ['FlowHierarchy']
Removed features by T-test (p < 0.05): ['SubgraphCentrality']
Kept features by T-test (p >= 0.05): ['EdgeCount', 'NodeCount', 'AverageDegree', 'GraphDensity', 'GraphRadius', 'Diameter', 'EdgeConnectivity', 'Modularity', 'CommunityEntropy', 'CommunityConductance', 'CommunityExpansion', 'AverageClusteringCoefficient', 'GlobalEfficiency', 'CharacteristicPathLength', 'AlgebraicConnectivity', 'Assortativity', 'Transitivity', 'LocalEfficiency', 'BetweennessCentrality', 'ClosenessCentrality', 'KatzCentrality', 'EigenvectorCentrality', 'HarmonicCentrality', 'LoadCentrality', 'TriangleCount', 'SquareCount', 'SpectralRadius', 'SpectralEntropy', 'GraphEnergy', 'NetworkEntropy', 'KConnectivity', 'CriticalThreshold', 'NumberOfCliques', 'GraphRobustness', 'WedgeCount', 'KCycleCount', 'Efficiency', 'ConstraintVariance', 'ArticulationPoints', 'Bridges', 'IntraCommunityDensity', 'InterCommunit

# combining data

In [6]:
import os
import re
import pandas as pd

# 1) Functions for extracting normalized IDs
def extract_functional_id(subject_id_str, label):
    """
    Extract numeric ID from functional data:
    - Functional IDs look like: resultsROI_Subject007_Condition001
    - Label: 1 => Patient => "sub###"
    - Label: 0 => Control => "subH###"
    """
    match = re.search(r"Subject(\d+)", subject_id_str)
    if not match:
        return None
    
    num_int = int(match.group(1))
    return f"sub{num_int}" if label == 1 else f"subH{num_int}"

def extract_structural_id(subject_id_str, label):
    """
    Extract numeric ID from structural data:
    - Structural IDs can look like: sub16, subH01, h1sub13
    - Additional patterns for diseased: team44OLF1sub*
    - Additional patterns for healthy: team44HC4sub*, team44HC1sub*
    - Label: 1 => Patient => "sub###"
    - Label: 0 => Control => "subH###"
    """
    if label == 1:
        # Patterns for diseased patients
        m_sub = re.match(r"^sub(\d+)$", subject_id_str, re.IGNORECASE)
        if m_sub:
            return f"sub{int(m_sub.group(1))}"
        m_hsub = re.match(r"^h\d*sub(\d+)$", subject_id_str, re.IGNORECASE)
        if m_hsub:
            return f"sub{int(m_hsub.group(1))}"
        m_olf = re.match(r"^team44OLF1sub(\d+)", subject_id_str, re.IGNORECASE)
        if m_olf:
            return f"sub{int(m_olf.group(1))}"
        return None
    else:
        # Patterns for healthy controls
        m_subH = re.match(r"^subH(\d+)$", subject_id_str, re.IGNORECASE)
        if m_subH:
            return f"subH{int(m_subH.group(1))}"
        m_hsub = re.match(r"^h\d*sub(\d+)$", subject_id_str, re.IGNORECASE)
        if m_hsub:
            return f"subH{int(m_hsub.group(1))}"
        m_hc4 = re.match(r"^team44HC4sub(\d+)", subject_id_str, re.IGNORECASE)
        if m_hc4:
            return f"subH{int(m_hc4.group(1))}"
        m_hc1 = re.match(r"^team44HC1sub(\d+)", subject_id_str, re.IGNORECASE)
        if m_hc1:
            return f"subH{int(m_hc1.group(1))}"
        return None


# 2) Combine structural and functional datasets
def combine_datasets(structural_csv, functional_csv):
    # Load structural dataset
    df_struct = pd.read_csv(structural_csv)
    # Normalize subject IDs in structural data
    df_struct['Normalized_ID'] = df_struct.apply(
        lambda row: extract_structural_id(row['Subject_ID'], row['Label']), axis=1
    )
    df_struct.dropna(subset=['Normalized_ID'], inplace=True)

    # Load functional dataset
    df_func = pd.read_csv(functional_csv)
    # Normalize subject IDs in functional data
    df_func['Normalized_ID'] = df_func.apply(
        lambda row: extract_functional_id(row['Subject_ID'], row['Label']), axis=1
    )
    df_func.dropna(subset=['Normalized_ID'], inplace=True)

    # Drop duplicates on normalized IDs
    df_struct = df_struct.drop_duplicates(subset=['Normalized_ID'])
    df_func = df_func.drop_duplicates(subset=['Normalized_ID'])

    # Merge on Normalized_ID
    merged = pd.merge(
        df_struct, df_func,
        how='inner',
        on='Normalized_ID',
        suffixes=('_stru', '_func')
    )

    # Drop redundant columns (e.g., duplicate `Label` or `Subject_ID` columns)
    merged = merged.drop(columns=['Normalized_ID', 'Subject_ID_func', 'Label_func']).rename(
        columns={'Subject_ID_stru': 'Subject_ID', 'Label_stru': 'Label'}
    )

    return merged

# 3) Example driver code
if __name__ == "__main__":
    # File paths
    functional_dataset_paths = [
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\putamen_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\all_no_put_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\table_only_subjects_removed.csv",
        r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_subjects_removed.csv",
        r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\whole_brain_conn_subjects_removed.csv"
    ]

    structural_dataset_paths = [
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\left_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\right_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\cleaned_mri_data_subjects_removed.csv",
    ]

    output_directory = r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined"
    os.makedirs(output_directory, exist_ok=True)

    for i, structural_path in enumerate(structural_dataset_paths, start=1):
        for j, functional_path in enumerate(functional_dataset_paths, start=1):
            combined_data = combine_datasets(structural_path, functional_path)
            output_path = os.path.join(
                output_directory,
                f"ttest_combined_structural_{i}_functional_{j}.csv"
            )
            combined_data.to_csv(output_path, index=False)
            print(f"Combined dataset saved to {output_path}")


Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_1_functional_1.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_1_functional_2.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_1_functional_3.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_1_functional_4.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_1_functional_5.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_2_functional_1.csv
Combined dataset saved to D:\image_group_data\team44\output_for_freesurfer_table\output\combined\ttest_combined_structural_2_functional_2.csv
Combin

# Machine learning 

In [19]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold


from sklearn.linear_model import (
    LogisticRegression, RidgeClassifier, SGDClassifier,
    Perceptron, PassiveAggressiveClassifier
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, BaggingClassifier, HistGradientBoostingClassifier
)
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier, NearestCentroid
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, confusion_matrix

# External Libraries Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Bayesian Optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# For handling class imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV

########################
# Define Custom Metrics
########################
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-9)

specificity_scorer = make_scorer(specificity_score, greater_is_better=True)

def supports_roc_auc(model):
    return hasattr(model, "predict_proba") or hasattr(model, "decision_function")

#def preprocess_for_lda(X):
    # Remove highly correlated features
#    def remove_highly_correlated_features(df, threshold=0.95):
 #       corr_matrix = df.corr().abs()
  #      upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
   #     to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    #    return df.drop(columns=to_drop, errors='ignore')
    
    # Step 1: Remove highly correlated features
#    X = remove_highly_correlated_features(X, threshold=0.95)
 #   
  #  # Step 2: Remove zero-variance features
   # from sklearn.feature_selection import VarianceThreshold
#    selector = VarianceThreshold(threshold=0.0)
 #   X = pd.DataFrame(selector.fit_transform(X), columns=X.columns[selector.get_support()])
    
    # Step 3: Apply PCA
  #  scaler = StandardScaler()
   # X_scaled = scaler.fit_transform(X)
#    pca = PCA(n_components=0.98)  # Retain 90% variance
 #   X_pca = pca.fit_transform(X_scaled)
    
  #  return X_pca



########################
# Define Models to Test
########################
models = {
    # Linear Models
   # "LogisticRegression": LogisticRegression(max_iter=1000),
   # "RidgeClassifier": RidgeClassifier(),
    "CalibratedRidge": CalibratedClassifierCV(
        estimator=RidgeClassifier(),
        cv=3
    ),
    "CalibratedPAC": CalibratedClassifierCV(
        estimator=PassiveAggressiveClassifier(max_iter=1000, random_state=42),
        cv=3
    ),


    "CalibratedSGD": CalibratedClassifierCV(
        estimator=SGDClassifier(max_iter=1000, random_state=42, loss='hinge'),
        cv=3,
        method='sigmoid'
    ),
    "CalibratedPerceptron": CalibratedClassifierCV(
        estimator=Perceptron(max_iter=1000, random_state=42),
        cv=3,
        method='sigmoid'
    ),
    "CalibratedPassiveAggressive": CalibratedClassifierCV(
        estimator=PassiveAggressiveClassifier(max_iter=1000, random_state=42),
        cv=3,                         # 3 or 5 folds for calibration
        method='sigmoid'             # typically 'sigmoid' for binary
    ),

    # Discriminant Analysis
    "LDA": LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    "QDA": QuadraticDiscriminantAnalysis(),

    # Naive Bayes
    "GaussianNB": GaussianNB(),
    #"MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB(),
    #"ComplementNB": ComplementNB(),

    # Tree-based
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(),
    "ExtraTrees": ExtraTreesClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),
    "HistGradientBoosting": HistGradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm='SAMME'),
    "Bagging": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=10),

    # SVM-based
    "SVC": SVC(probability=True),
    "CalibratedLinearSVC": CalibratedClassifierCV(
        estimator=LinearSVC(max_iter=10000),
        cv=3,
        method='sigmoid'
    ),
    "NuSVC": NuSVC(probability=True),

    # Neighbors and Centroid-based
    "KNN": KNeighborsClassifier(),
    #"RadiusNeighborsClassifier": RadiusNeighborsClassifier(radius=1.0),
#    "CalibratedNearestCentroid": CalibratedClassifierCV(
 #       estimator=NearestCentroid(),
  #      cv=3,
   #     method='sigmoid'
    #),

    # Gaussian Process
    "GaussianProcessClassifier": GaussianProcessClassifier(random_state=42),

    # Neural Network (Shallow)
    "MLPClassifier": MLPClassifier(max_iter=2000, random_state=42),

    # External Models
    "XGBClassifier": XGBClassifier(eval_metric='logloss'),
    "LGBMClassifier": LGBMClassifier(),
    "CatBoostClassifier": CatBoostClassifier(verbose=0)
}


########################
# Bayesian Optimization Parameter Spaces
########################
from skopt.space import Real, Integer, Categorical

param_spaces = {
    "LogisticRegression": {
        'classifier__C': Real(1e-4, 100, prior='log-uniform'),
        'classifier__solver': Categorical(['lbfgs', 'liblinear', 'sag', 'saga'])
    },
    "CalibratedRidge": {
        'classifier__estimator__alpha': Real(1e-4, 10, prior='log-uniform')
    },
    "CalibratedPAC": {
        'classifier__estimator__C': Real(1e-3, 10, prior='log-uniform')
    },
    "CalibratedSGD": {
        # Because SGD can tune alpha, loss, etc.
        'classifier__estimator__alpha': Real(1e-6, 1, prior='log-uniform'),
        'classifier__estimator__loss': Categorical(['hinge','log','modified_huber','squared_hinge','perceptron'])
    },
    "CalibratedPerceptron": {
        'classifier__estimator__alpha': Real(1e-6, 1, prior='log-uniform')
    },
    "CalibratedLinearSVC": {
        'classifier__estimator__C': Real(1e-3, 10, prior='log-uniform')
    },

    "LDA": {
        'classifier__shrinkage': Categorical(['auto', 0.1, 0.5, 0.9, None]),  # Valid shrinkage values
        'classifier__solver': Categorical(['lsqr']),  # Restrict to 'lsqr' since shrinkage requires it
    },

    "QDA": {
        'classifier__reg_param': Real(0, 1, prior='uniform')
    },
    "GaussianNB": {
        'classifier__var_smoothing': Real(1e-12, 1e-3, prior='log-uniform')
    },
  #  "MultinomialNB": {
   #     'classifier__alpha': Real(1e-3, 10, prior='log-uniform')
   # },
    "BernoulliNB": {
        'classifier__alpha': Real(1e-3, 10, prior='log-uniform')
    },
    #"ComplementNB": {
    #    'classifier__alpha': Real(1e-3, 10, prior='log-uniform')
    #},
    "DecisionTree": {
        'classifier__max_depth': Integer(1, 50),
        'classifier__min_samples_split': Integer(2, 20),
        'classifier__min_samples_leaf': Integer(1, 20)
    },
    "RandomForest": {
        'classifier__n_estimators': Integer(50, 300),
        'classifier__max_depth': Integer(2, 50),
        'classifier__min_samples_split': Integer(2, 20)
    },
    "ExtraTrees": {
        'classifier__n_estimators': Integer(50, 300),
        'classifier__max_depth': Integer(2, 50),
        'classifier__min_samples_split': Integer(2, 20)
    },
    "GradientBoosting": {
        'classifier__n_estimators': Integer(50, 300),
        'classifier__learning_rate': Real(1e-3, 0.3, prior='log-uniform'),
        'classifier__max_depth': Integer(1, 10)
    },
    "HistGradientBoosting": {
        'classifier__max_iter': Integer(50, 300),
        'classifier__learning_rate': Real(1e-3, 0.3, prior='log-uniform'),
        'classifier__max_depth': Integer(1, 10)
    },
    "AdaBoost": {
        'classifier__n_estimators': Integer(50, 300),
        'classifier__learning_rate': Real(1e-3, 1, prior='log-uniform')
    },
    "Bagging": {
        'classifier__n_estimators': Integer(10, 200),
        'classifier__max_samples': Real(0.1, 1.0, prior='uniform')
    },
    "SVC": {
        'classifier__C': Real(1e-3, 10, prior='log-uniform'),
        'classifier__gamma': Real(1e-4, 1, prior='log-uniform'),
        'classifier__kernel': Categorical(['linear', 'rbf', 'poly', 'sigmoid'])
    },
#    "LinearSVC": {
 #       'classifier__C': Real(1e-3, 10, prior='log-uniform')
  #  },
    "NuSVC": {
        'classifier__nu': Real(0.01, 0.9, prior='uniform'),
        'classifier__gamma': Real(1e-4, 1, prior='log-uniform'),
        'classifier__kernel': Categorical(['linear', 'rbf', 'poly', 'sigmoid'])
    },
    "KNN": {
        'classifier__n_neighbors': Integer(1, 50),
        'classifier__leaf_size': Integer(10, 50)
    },
    "RadiusNeighborsClassifier": {
        'classifier__radius': Real(0.1, 10, prior='log-uniform')
    },
#    "CalibratedNearestCentroid": {
 #       'classifier__estimator__shrink_threshold': Real(0.0, 2.0, prior='uniform'),
  #      'classifier__cv': Integer(2, 10),
   #     'classifier__method': Categorical(['sigmoid', 'isotonic'])
    #},
    "GaussianProcessClassifier": {
        # GPC is complex because of kernels, we'll just optimize the max_iter_predict
        'classifier__max_iter_predict': Integer(50, 300)
    },
    "MLPClassifier": {
        'classifier__alpha': Real(1e-6, 1e-1, prior='log-uniform'),
        'classifier__learning_rate_init': Real(1e-4, 0.1, prior='log-uniform'),
        'classifier__hidden_layer_sizes': Integer(50, 300)  # single hidden layer size
    },
    "XGBClassifier": {
        'classifier__n_estimators': Integer(50, 300),
        'classifier__learning_rate': Real(1e-3, 0.3, prior='log-uniform'),
        'classifier__max_depth': Integer(1,10)
    },
    "LGBMClassifier": {
    'classifier__n_estimators': Integer(50, 300),        # Number of trees
    'classifier__learning_rate': Real(1e-3, 0.3, prior='log-uniform'),  # Learning rate
    'classifier__num_leaves': Integer(10, 100),         # Number of leaves (increase range for flexibility)
    'classifier__max_depth': Integer(-1, 20),           # Tree depth (-1 for unlimited)
    'classifier__min_child_samples': Integer(1, 20),    # Minimum samples per leaf
    'classifier__min_split_gain': Real(0.0, 1.0, prior='uniform'),  # Minimum gain for split
    'classifier__colsample_bytree': Real(0.6, 1.0, prior='uniform'),  # Feature fraction for each tree
    'classifier__lambda_l1': Real(1e-6, 10.0, prior='log-uniform'),    # L1 regularization (avoid 0)
    'classifier__lambda_l2': Real(1e-6, 10.0, prior='log-uniform')     # L2 regularization (avoid 0)
    },


    "CatBoostClassifier": {
        'classifier__iterations': Integer(50, 300),
        'classifier__learning_rate': Real(1e-3, 0.3, prior='log-uniform'),
        'classifier__depth': Integer(1, 10)
    }
}


########################
# Cross-Validation & PCA Settings
########################
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
pca_components = 0.95

########################
# Scoring Metrics
########################
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': specificity_scorer,
    'roc_auc': 'roc_auc'
}

########################
# Main Code
########################

def process_datasets(dataset_paths, use_bayes_opt=True, n_iter=10):
    results = []

    for dataset_path in dataset_paths:
        dataset_name = os.path.splitext(os.path.basename(dataset_path))[0]
        df = pd.read_csv(dataset_path)
        print(dataset_path, df['Label'].value_counts())
        # Remove Subject_ID if present
        if 'Subject_ID' in df.columns:
            df = df.drop(columns=['Subject_ID'])

        X = df.drop(columns=['Label'])
        y = df['Label']
        
        # Split the dataset into Train+Validation and Test sets
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print("Training set label counts:", y_train_val.value_counts())
        
        # Check for imbalance
        class_counts = y_train_val.value_counts()
        majority_class_count = class_counts.max()
        minority_class_count = class_counts.min()
        imbalance_ratio = majority_class_count / minority_class_count
        apply_smote = imbalance_ratio > 1.5  # Arbitrary threshold, adjust as needed
        print("imbalance_ratio:", imbalance_ratio)

        # Base pipeline steps (for non-LDA, non-NB models)
        base_steps = []
        if apply_smote:
            base_steps.append(('scaler', StandardScaler()))
            base_steps.append(('smote', SMOTE(random_state=42)))
            base_steps.append(('pca', PCA(n_components=pca_components)))
        else:
            base_steps.append(('scaler', StandardScaler()))
            base_steps.append(('pca', PCA(n_components=pca_components)))

        # Custom transformer to remove highly correlated features
        from sklearn.base import BaseEstimator, TransformerMixin
        class CorrelationRemover(BaseEstimator, TransformerMixin):
            def __init__(self, threshold=0.95):
                self.threshold = threshold
                self.to_drop_ = None
            def fit(self, X, y=None):
                df_corr = pd.DataFrame(X)
                corr_matrix = df_corr.corr().abs()
                upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
                return self
            def transform(self, X, y=None):
                df = pd.DataFrame(X)
                return df.drop(columns=self.to_drop_, errors='ignore').values

        # Preprocessing pipeline for LDA
        lda_preprocessing = Pipeline([
            ('corr_remove', CorrelationRemover(threshold=0.80)),
            ('var_thresh', VarianceThreshold(threshold=0.0)),
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.90))  # Adjust variance retained as needed
        ])

        # Fit LDA preprocessing pipeline on the training set and transform both sets
        #lda_preprocessing.fit(X_train_val, y_train_val)
        #X_train_val_preprocessed = lda_preprocessing.transform(X_train_val)
        #X_test_preprocessed = lda_preprocessing.transform(X_test)

        for model_name, model in models.items():
            # 1) Build pipeline steps (common to all models)
            pipeline_steps = [
                ('corr_remove', CorrelationRemover(threshold=0.95)),
                ('var_thresh', VarianceThreshold(threshold=0.0)),
                ('scaler', StandardScaler()),
                ('pca', PCA(n_components=pca_components)),
                ('classifier', model)
            ]
    
            # 2) Insert SMOTE if imbalanced
            if apply_smote:
                pipeline_steps.insert(2, ('smote', SMOTE(random_state=42)))
    
            # 3) Build pipeline
            pipeline = ImbPipeline(steps=pipeline_steps)
    
            # 4) If Bayesian optimization
            if use_bayes_opt and model_name in param_spaces:
                bayes_search = BayesSearchCV(
                    estimator=pipeline,
                    search_spaces=param_spaces[model_name],
                    scoring='accuracy',  # or "f1", "accuracy", ...
                    n_iter=n_iter,
                    cv=cv,
                    n_jobs=-1,
                    random_state=42,
                    verbose=0
                )
                bayes_search.fit(X_train_val, y_train_val)
                best_pipeline = bayes_search.best_estimator_
                
                # Evaluate via cross_validate
                cv_results = cross_validate(
                    best_pipeline, 
                    X_train_val, 
                    y_train_val,
                    cv=cv, 
                    scoring=scoring,
                    n_jobs=-1
                )
                # Final fit
                best_pipeline.fit(X_train_val, y_train_val)
                test_accuracy = best_pipeline.score(X_test, y_test)
        
            else:
                # No bayes opt
                cv_results = cross_validate(
                    pipeline, 
                    X_train_val, 
                    y_train_val,
                    cv=cv, 
                    scoring=scoring,
                    n_jobs=-1
                )
                pipeline.fit(X_train_val, y_train_val)
                test_accuracy = pipeline.score(X_test, y_test)
        
            # 5) Extract metrics
            mean_accuracy = np.mean(cv_results['test_accuracy'])
            mean_f1 = np.mean(cv_results['test_f1'])
            mean_precision = np.mean(cv_results['test_precision'])
            mean_recall = np.mean(cv_results['test_recall'])
            mean_specificity = np.mean(cv_results['test_specificity'])
    
            # AUC-ROC (if it exists in the dict)
            mean_auc_roc = None
            if 'test_roc_auc' in cv_results:
                mean_auc_roc = np.mean(cv_results['test_roc_auc'])
            
            # 6) Append results
            results.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Imbalance_Ratio': imbalance_ratio,
                'SMOTE_Used': apply_smote,
                'CV_Accuracy': mean_accuracy,
                'Test_Accuracy': test_accuracy,
                'F1': mean_f1,
                'Precision': mean_precision,
                'Recall': mean_recall,
                'Specificity': mean_specificity,
                'AUC-ROC': mean_auc_roc
            })


    results_df = pd.DataFrame(results)
    return results_df


########################
# Example Usage
########################
if __name__ == "__main__":
    dataset_paths = [
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\putamen_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\all_no_put_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\table_only_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\left_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\right_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\cleaned_mri_data_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_5.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_5.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_5.csv",
    ]

    results_df = process_datasets(dataset_paths, use_bayes_opt=True, n_iter=10)
    results_df.to_csv(r"D:\image_group_data\team44\output_for_freesurfer_table\result\ML_all_ttest_bayesopt_smote.csv", index=False)
    print(results_df)


D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\putamen_subjects_removed.csv Label
0    64
1    31
Name: count, dtype: int64
Training set label counts: Label
0    51
1    25
Name: count, dtype: int64
imbalance_ratio: 2.04
[LightGBM] [Warning] lambda_l1 is set=1.5247791391944723, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.5247791391944723
[LightGBM] [Warning] lambda_l2 is set=0.00013300585802877296, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.00013300585802877296
[LightGBM] [Warning] lambda_l1 is set=1.5247791391944723, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.5247791391944723
[LightGBM] [Warning] lambda_l2 is set=0.00013300585802877296, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.00013300585802877296
[LightGBM] [Info] Number of positive: 51, number of negative: 51
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000041 seconds.
You can set `force_col_wise=

C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


[LightGBM] [Warning] lambda_l1 is set=0.26763977234469377, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.26763977234469377
[LightGBM] [Warning] lambda_l2 is set=0.0003259571586662193, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0003259571586662193
[LightGBM] [Warning] lambda_l1 is set=0.26763977234469377, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.26763977234469377
[LightGBM] [Warning] lambda_l2 is set=0.0003259571586662193, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0003259571586662193
[LightGBM] [Info] Number of positive: 50, number of negative: 50
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000050 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1190
[LightGBM] [Info] Number of data points in the train set: 100, number of used features: 34
[LightGBM] [Info] [binary:BoostFro

C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


[LightGBM] [Warning] lambda_l1 is set=0.0011646737978730447, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0011646737978730447
[LightGBM] [Warning] lambda_l2 is set=0.004856704273115606, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.004856704273115606
[LightGBM] [Warning] lambda_l1 is set=0.0011646737978730447, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0011646737978730447
[LightGBM] [Warning] lambda_l2 is set=0.004856704273115606, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.004856704273115606
[LightGBM] [Info] Number of positive: 50, number of negative: 50
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1155
[LightGBM] [Info] Number of data points in the train set: 100, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [War

...

[LightGBM] [Warning] lambda_l1 is set=0.0011646737978730447, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0011646737978730447
[LightGBM] [Warning] lambda_l2 is set=0.004856704273115606, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.004856704273115606
[LightGBM] [Warning] lambda_l1 is set=0.0011646737978730447, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0011646737978730447
[LightGBM] [Warning] lambda_l2 is set=0.004856704273115606, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.004856704273115606
[LightGBM] [Info] Number of positive: 50, number of negative: 50
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000413 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1820
[LightGBM] [Info] Number of data points in the train set: 100, number of used features: 52
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [War

C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


[LightGBM] [Warning] lambda_l1 is set=0.26763977234469377, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.26763977234469377
[LightGBM] [Warning] lambda_l2 is set=0.0003259571586662193, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0003259571586662193
[LightGBM] [Warning] lambda_l1 is set=0.26763977234469377, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.26763977234469377
[LightGBM] [Warning] lambda_l2 is set=0.0003259571586662193, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0003259571586662193
[LightGBM] [Info] Number of positive: 32, number of negative: 32
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000298 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 943
[LightGBM] [Info] Number of data points in the train set: 64, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] 

C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


[LightGBM] [Warning] lambda_l1 is set=2.69809757549238, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.69809757549238
[LightGBM] [Warning] lambda_l2 is set=5.420184998488358e-06, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.420184998488358e-06
[LightGBM] [Warning] lambda_l1 is set=2.69809757549238, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.69809757549238
[LightGBM] [Warning] lambda_l2 is set=5.420184998488358e-06, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.420184998488358e-06
[LightGBM] [Info] Number of positive: 32, number of negative: 32
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 943
[LightGBM] [Info] Number of data points in the train set: 64, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further s

# Results for Linear Discriminant Analysis (LDA) and LightGBM are not valid

In [20]:
results_df

,Dataset,Model,Imbalance_Ratio,SMOTE_Used,CV_Accuracy,Test_Accuracy,F1,Precision,Recall,Specificity,AUC-ROC
0,putamen_subjects_removed,CalibratedRidge,2.04,True,0.700000,0.368421,0.574286,0.596667,0.633333,0.750000,0.761667
1,putamen_subjects_removed,CalibratedPAC,2.04,True,0.687500,0.526316,0.587143,0.538333,0.716667,0.690000,0.808333
2,putamen_subjects_removed,CalibratedSGD,2.04,True,0.700000,0.368421,0.580476,0.521667,0.716667,0.710000,0.821667
3,putamen_subjects_removed,CalibratedPerceptron,2.04,True,0.630357,0.473684,0.453810,0.448333,0.500000,0.710000,0.725000
4,putamen_subjects_removed,CalibratedPassiveAggressive,2.04,True,0.655357,0.526316,0.496190,0.465000,0.566667,0.710000,0.685000
...,...,...,...,...,...,...,...,...,...,...,...
520,combined_structural_3_functional_5,GaussianProcessClassifier,1.60,True,0.786667,0.785714,0.670000,0.716667,0.700000,0.850000,0.500000
521,combined_structural_3_functional_5,MLPClassifier,1.60,True,0.860000,0.928571,0.800000,0.900000,0.750000,0.933333,0.916667
522,combined_structural_3_functional_5,XGBClassifier,1.60,True,0.813333,0.642857,0.696667,0.716667,0.700000,0.875000,0.758333
523,combined_structural_3_functional_5,LGBMClassifier,1.60,True,0.793333,0.642857,0.730000,0.716667,0.750000,0.816667,0.841667


# Deep learning 

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    make_scorer
)
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, InputLayer, Conv1D, Flatten,
    LSTM, GRU, BatchNormalization, MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Bayesian Optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical


###############################################################################
# 1) Custom Specificity Scorer & Scoring Dict
###############################################################################
def specificity_score(y_true, y_pred):
    """TN/(TN+FP) for binary classification."""
    if y_pred.dtype not in [np.int32, np.int64, np.bool_]:
        y_pred = (y_pred >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-9)

specificity_scorer = make_scorer(specificity_score, greater_is_better=True)

scoring = {
    'accuracy':    make_scorer(accuracy_score),
    'f1':          make_scorer(f1_score, average='binary'),
    'precision':   make_scorer(precision_score, average='binary', zero_division=0),
    'recall':      make_scorer(recall_score, average='binary'),
    'specificity': specificity_scorer,
    'roc_auc':     make_scorer(roc_auc_score, needs_proba=True)
}


###############################################################################
# 2) Model Builders (Using meta=None)
###############################################################################
from tensorflow.keras.regularizers import l2

def build_mlp_simple_model(lr=1e-3, hidden_units=64, dropout_rate=0.4, meta=None):
    input_dim = meta["n_features_in_"]
    model = Sequential([
        InputLayer(input_shape=(input_dim,)),
        Dense(hidden_units, activation='relu', kernel_regularizer=l2(0.01)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_mlp_deeper_model(
    lr=1e-3,
    hidden_units_1=128,
    hidden_units_2=64,
    hidden_units_3=32,
    dropout_rate=0.5,
    meta=None
):
    input_dim = meta["n_features_in_"]
    model = Sequential([
        InputLayer(input_shape=(input_dim,)),
        Dense(hidden_units_1, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(hidden_units_2, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(hidden_units_3, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(1, activation='sigmoid')
    ])
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_1d_model(
    lr=1e-3,
    filters=32,
    kernel_size=3,
    hidden_units=64,
    dropout_rate=0.3,
    meta=None
):
    input_dim = meta["n_features_in_"]
    model = Sequential([
        InputLayer(input_shape=(input_dim, 1)),
        Conv1D(filters=filters, kernel_size=kernel_size, activation='relu'),
        Flatten(),
        Dense(hidden_units, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_gru_model(lr=1e-3, gru_units=64, dropout_rate=0.3, meta=None):
    input_dim = meta["n_features_in_"]
    model = Sequential([
        InputLayer(input_shape=(input_dim, 1)),
        GRU(gru_units, activation='tanh'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

from tensorflow.keras.layers import Input, MultiHeadAttention, Dense, Dropout, Flatten, LayerNormalization

def build_transformer_model(lr=1e-3, num_heads=4, hidden_units=64, dropout_rate=0.4, meta=None):
    input_dim = meta["n_features_in_"]
    
    # Input layer
    inputs = Input(shape=(input_dim, 1))
    
    # MultiHeadAttention layer
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=hidden_units)(inputs, inputs)
    attention_output = LayerNormalization()(attention_output)
    
    # Fully connected layers
    dense_output = Dense(hidden_units, activation='relu')(attention_output)
    dense_output = Dropout(dropout_rate)(dense_output)
    
    # Flatten the output before the final layer
    flat_output = Flatten()(dense_output)
    outputs = Dense(1, activation='sigmoid')(flat_output)
    
    # Define the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    # Compile the model
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model



###############################################################################
# 3) Model Config Dictionary
###############################################################################
# We'll add a "reshape_to_3d" step if reshape_needed is True.
reshape_to_3d = FunctionTransformer(lambda X: np.expand_dims(X, axis=-1))

model_configs = {
    "MLP_Simple": {
        "build_fn": build_mlp_simple_model,
        "reshape_needed": False,
        "param_space": {
            "clf__model__lr": Real(1e-5, 1e-2, prior='log-uniform'),
            "clf__model__hidden_units": Integer(32, 256),
            "clf__model__dropout_rate": Real(0.0, 0.7),
            "clf__epochs": Integer(5, 50),
            "clf__batch_size": Integer(16, 128)
        }
    },
    "MLP_Deeper": {
        "build_fn": build_mlp_deeper_model,
        "reshape_needed": False,
        "param_space": {
            "clf__model__lr": Real(1e-5, 1e-2, prior='log-uniform'),
            "clf__model__hidden_units_1": Integer(64, 256),
            "clf__model__hidden_units_2": Integer(32, 128),
            "clf__model__hidden_units_3": Integer(16, 64),
            "clf__model__dropout_rate": Real(0.0, 0.7),
            "clf__epochs": Integer(5, 50),
            "clf__batch_size": Integer(16, 128)
        }
    },
    "CNN_1D": {
        "build_fn": build_cnn_1d_model,
        "reshape_needed": True,
        "param_space": {
            "clf__model__lr": Real(1e-5, 1e-2, prior='log-uniform'),
            "clf__model__filters": Integer(8, 64),
            "clf__model__kernel_size": Integer(2, 5),
            "clf__model__hidden_units": Integer(32, 256),
            "clf__model__dropout_rate": Real(0.0, 0.7),
            "clf__epochs": Integer(5, 50),
            "clf__batch_size": Integer(16, 128)
        }
    },
    "GRU": {
        "build_fn": build_gru_model,
        "reshape_needed": True,
        "param_space": {
            "clf__model__lr": Real(1e-5, 1e-2, prior='log-uniform'),
            "clf__model__gru_units": Integer(16, 128),
            "clf__model__dropout_rate": Real(0.0, 0.7),
            "clf__epochs": Integer(5, 50),
            "clf__batch_size": Integer(16, 128)
        }
    },
    "Transformer": {
        "build_fn": build_transformer_model,
        "reshape_needed": True,
        "param_space": {
            "clf__model__lr": Real(1e-5, 1e-2, prior='log-uniform'),
            "clf__model__num_heads": Integer(2, 8),
            "clf__model__hidden_units": Integer(32, 256),
            "clf__model__dropout_rate": Real(0.0, 0.7),
            "clf__epochs": Integer(5, 50),
            "clf__batch_size": Integer(16, 128)
        }
    }
}


###############################################################################
# 4) Main Function to Evaluate All Models on One CSV
###############################################################################
def evaluate_models_on_dataset(csv_path):
    df = pd.read_csv(csv_path)
    if 'Subject_ID' in df.columns:
        df.drop(columns=['Subject_ID'], inplace=True)
    X = df.drop(columns=['Label'])
    y = df['Label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    results = []

    for model_name, cfg in model_configs.items():
        print(f"=== Running Model: {model_name} on dataset: {os.path.basename(csv_path)} ===")

        build_fn       = cfg["build_fn"]
        reshape_needed = cfg["reshape_needed"]
        param_space    = cfg["param_space"]

        keras_estimator = KerasClassifier(
            model=build_fn,
            verbose=0,
            validation_split=0.2,
            callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
        )

        # Build pipeline steps
        steps = [
            ("smote", SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=0.95)),
        ]

        # If reshape_needed, add the "reshape_to_3d" step
        if reshape_needed:
            steps.append(("reshape", reshape_to_3d))

        # Finally, add the Keras classifier
        steps.append(("clf", keras_estimator))

        pipeline = Pipeline(steps)

        bayes_search = BayesSearchCV(
            estimator=pipeline,
            search_spaces=param_space,
            n_iter=10,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring='roc_auc',
            n_jobs=-1,
            refit=True,
            verbose=0,
            random_state=42
        )

        bayes_search.fit(X_train, y_train)

        best_pipeline = bayes_search.best_estimator_
        # Training metrics
        y_train_pred_proba = best_pipeline.predict_proba(X_train)[:, 1]
        y_train_pred = (y_train_pred_proba >= 0.5).astype(int)
        tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train, y_train_pred).ravel()

        train_metrics = {
            "train_accuracy":  accuracy_score(y_train, y_train_pred),
            "train_f1":        f1_score(y_train, y_train_pred),
            "train_precision": precision_score(y_train, y_train_pred, zero_division=0),
            "train_recall":    recall_score(y_train, y_train_pred),
            "train_specificity": tn_train / (tn_train + fp_train + 1e-9),
            "train_roc_auc":   roc_auc_score(y_train, y_train_pred_proba),
        }
        
        # Print training metrics
        print(f"Training Metrics for {model_name}: {train_metrics}")
        print(f"  Best Params for {model_name}: {bayes_search.best_params_}")

        # Predict directly from the final pipeline
        # This ensures the pipeline's "reshape" step is used on the test data
        y_test_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
        y_test_pred = (y_test_pred_proba >= 0.5).astype(int)

        test_metrics = {}
        test_metrics["test_accuracy"]  = accuracy_score(y_test, y_test_pred)
        test_metrics["test_f1"]        = f1_score(y_test, y_test_pred)
        test_metrics["test_precision"] = precision_score(y_test, y_test_pred, zero_division=0)
        test_metrics["test_recall"]    = recall_score(y_test, y_test_pred)
        tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
        specificity = tn / (tn + fp + 1e-9)
        test_metrics["test_specificity"] = specificity
        test_metrics["test_roc_auc"]  = roc_auc_score(y_test, y_test_pred_proba)
        print(f"Test Metrics for {model_name}: {test_metrics}")
        
        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_test_pred)
        print(f"Confusion Matrix for {model_name}:\n{conf_matrix}")

        # Cross-validation results
        cv_results = bayes_search.cv_results_
        mean_cv_score = cv_results['mean_test_score'].max()
        print(f"Mean CV ROC-AUC for {model_name}: {mean_cv_score}")

        row = {
            "Dataset":   os.path.basename(csv_path),
            "Model":     model_name,
            "BestParams": bayes_search.best_params_,
            "conf_matrix": conf_matrix.tolist(),  # Convert confusion matrix to list for saving
            "mean_cv_roc_auc": mean_cv_score,
        }
        row.update(train_metrics)
        row.update(test_metrics)
        results.append(row)


    return pd.DataFrame(results)


###############################################################################
# 6) Example Usage for Multiple CSV Datasets
###############################################################################
if __name__ == "__main__":
    dataset_paths = [
        r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\putamen_subjects_removed.csv",
        r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\all_no_put_subjects_removed.csv",
        r"D:\image_group_data\team44\CONN_Team44\all_first_levlels\optimize_output\table_only_subjects_removed.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\left_hemisphere_mri_data_filtered.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\right_hemisphere_mri_data_filtered.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\cleaned_mri_data_subjects_removed.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_1_functional_1.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_1_functional_2.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_1_functional_3.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_1_functional_4.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_1_functional_5.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_2_functional_1.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_2_functional_2.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_2_functional_3.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_2_functional_4.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_2_functional_5.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_3_functional_1.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_3_functional_2.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_3_functional_3.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_3_functional_4.csv",
        r"D:\image_group_data\team44\output_for_freesurfer_table\output\combined\combined_structural_3_functional_5.csv",
    ]

    all_results = []
    for dpath in dataset_paths:
        df_res = evaluate_models_on_dataset(dpath)
        all_results.append(df_res)

    final_results = pd.concat(all_results, ignore_index=True)
    print(final_results)

    output_csv = r"D:\image_group_data\team44\output_for_freesurfer_table\result\DL_all_ttest_bayesopt_smote_overfit.csv"
    final_results.to_csv(output_csv, index=False)
    print(f"Results saved to: {output_csv}")


C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\sklearn\metrics\_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


=== Running Model: MLP_Simple on dataset: putamen_subjects_removed.csv ===


C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Training Metrics for MLP_Simple: {'train_accuracy': 0.8289473684210527, 'train_f1': 0.7346938775510204, 'train_precision': 0.75, 'train_recall': 0.72, 'train_specificity': 0.8823529411591696, 'train_roc_auc': 0.9113725490196078}
  Best Params for MLP_Simple: OrderedDict({'clf__batch_size': 106, 'clf__epochs': 25, 'clf__model__dropout_rate': 0.3686341659893847, 'clf__model__hidden_units': 192, 'clf__model__lr': 0.005147024286785697})
Test Metrics for MLP_Simple: {'test_accuracy': 0.5263157894736842, 'test_f1': 0.18181818181818182, 'test_precision': 0.2, 'test_recall': 0.16666666666666666, 'test_specificity': 0.6923076922544379, 'test_roc_auc': 0.3333333333333333}
Confusion Matrix for MLP_Simple:
[[9 4]
 [5 1]]
Mean CV ROC-AUC for MLP_Simple: 0.7589090909090909
=== Running Model: MLP_Deeper on dataset: putamen_subjects_removed.csv ===


C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


...

Training Metrics for CNN_1D: {'train_accuracy': 0.5394736842105263, 'train_f1': 0.2222222222222222, 'train_precision': 0.25, 'train_recall': 0.2, 'train_specificity': 0.7058823529273357, 'train_roc_auc': 0.47372549019607846}
  Best Params for CNN_1D: OrderedDict({'clf__batch_size': 98, 'clf__epochs': 47, 'clf__model__dropout_rate': 0.11452502504698517, 'clf__model__filters': 19, 'clf__model__hidden_units': 211, 'clf__model__kernel_size': 3, 'clf__model__lr': 0.0002382722920147222})
Test Metrics for CNN_1D: {'test_accuracy': 0.631578947368421, 'test_f1': 0.2222222222222222, 'test_precision': 0.3333333333333333, 'test_recall': 0.16666666666666666, 'test_specificity': 0.8461538460887574, 'test_roc_auc': 0.37179487179487175}
Confusion Matrix for CNN_1D:
[[11  2]
 [ 5  1]]
Mean CV ROC-AUC for CNN_1D: 0.6905454545454546
=== Running Model: GRU on dataset: all_no_put_subjects_removed.csv ===


C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Training Metrics for GRU: {'train_accuracy': 0.618421052631579, 'train_f1': 0.0, 'train_precision': 0.0, 'train_recall': 0.0, 'train_specificity': 0.9215686274329105, 'train_roc_auc': 0.523921568627451}
  Best Params for GRU: OrderedDict({'clf__batch_size': 98, 'clf__epochs': 47, 'clf__model__dropout_rate': 0.11452502504698517, 'clf__model__gru_units': 37, 'clf__model__lr': 0.0025206334448741735})
Test Metrics for GRU: {'test_accuracy': 0.631578947368421, 'test_f1': 0.36363636363636365, 'test_precision': 0.4, 'test_recall': 0.3333333333333333, 'test_specificity': 0.7692307691715976, 'test_roc_auc': 0.41025641025641024}
Confusion Matrix for GRU:
[[10  3]
 [ 4  2]]
Mean CV ROC-AUC for GRU: 0.6487272727272727
=== Running Model: Transformer on dataset: all_no_put_subjects_removed.csv ===
Training Metrics for Transformer: {'train_accuracy': 0.6710526315789473, 'train_f1': 0.0, 'train_precision': 0.0, 'train_recall': 0.0, 'train_specificity': 0.9999999999803922, 'train_roc_auc': 0.5}
  Best 

C:\Users\Growth fire\PycharmProjects\pythonProject\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Training Metrics for MLP_Simple: {'train_accuracy': 0.5657894736842105, 'train_f1': 0.35294117647058826, 'train_precision': 0.34615384615384615, 'train_recall': 0.36, 'train_specificity': 0.6666666666535949, 'train_roc_auc': 0.52}
  Best Params for MLP_Simple: OrderedDict({'clf__batch_size': 123, 'clf__epochs': 37, 'clf__model__dropout_rate': 0.6101563499242505, 'clf__model__hidden_units': 125, 'clf__model__lr': 0.0001390574606467376})
Test Metrics for MLP_Simple: {'test_accuracy': 0.3684210526315789, 'test_f1': 0.14285714285714285, 'test_precision': 0.125, 'test_recall': 0.16666666666666666, 'test_specificity': 0.4615384615029586, 'test_roc_auc': 0.2564102564102564}
Confusion Matrix for MLP_Simple:
[[6 7]
 [5 1]]
Mean CV ROC-AUC for MLP_Simple: 0.5629090909090909
=== Running Model: MLP_Deeper on dataset: table_only_subjects_removed.csv ===


In [40]:
final_results

,Dataset,Model,BestParams,conf_matrix,mean_cv_roc_auc,train_accuracy,train_f1,train_precision,train_recall,train_specificity,train_roc_auc,test_accuracy,test_f1,test_precision,test_recall,test_specificity,test_roc_auc
0,putamen_subjects_removed.csv,MLP_Simple,"{'clf__batch_size': 16, 'clf__epochs': 42, 'cl...","[[10, 3], [6, 0]]",0.641612,0.776316,0.585366,0.750000,0.48,0.921569,0.857255,0.526316,0.000000,0.000000,0.000000,0.769231,0.346154
1,putamen_subjects_removed.csv,MLP_Deeper,"{'clf__batch_size': 16, 'clf__epochs': 42, 'cl...","[[7, 6], [6, 0]]",0.647331,0.684211,0.612903,0.513514,0.76,0.647059,0.829804,0.368421,0.000000,0.000000,0.000000,0.538462,0.320513
2,putamen_subjects_removed.csv,CNN_1D,"{'clf__batch_size': 16, 'clf__epochs': 42, 'cl...","[[5, 8], [2, 4]]",0.628813,0.552632,0.514286,0.400000,0.72,0.470588,0.631373,0.473684,0.444444,0.333333,0.666667,0.384615,0.525641
3,putamen_subjects_removed.csv,GRU,"{'clf__batch_size': 62, 'clf__epochs': 38, 'cl...","[[9, 4], [5, 1]]",0.662854,0.552632,0.227273,0.263158,0.20,0.725490,0.525490,0.526316,0.181818,0.200000,0.166667,0.692308,0.320513
4,putamen_subjects_removed.csv,Transformer,"{'clf__batch_size': 62, 'clf__epochs': 38, 'cl...","[[13, 0], [6, 0]]",0.500000,0.671053,0.000000,0.000000,0.00,1.000000,0.500000,0.684211,0.000000,0.000000,0.000000,1.000000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,combined_structural_3_functional_5.csv,MLP_Simple,"{'clf__batch_size': 16, 'clf__epochs': 42, 'cl...","[[9, 0], [0, 5]]",0.901154,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
101,combined_structural_3_functional_5.csv,MLP_Deeper,"{'clf__batch_size': 85, 'clf__epochs': 40, 'cl...","[[8, 1], [1, 4]]",0.852814,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,0.857143,0.800000,0.800000,0.800000,0.888889,0.933333
102,combined_structural_3_functional_5.csv,CNN_1D,"{'clf__batch_size': 77, 'clf__epochs': 46, 'cl...","[[9, 0], [0, 5]]",0.912698,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
103,combined_structural_3_functional_5.csv,GRU,"{'clf__batch_size': 85, 'clf__epochs': 40, 'cl...","[[3, 6], [4, 1]]",0.562193,0.576923,0.476190,0.454545,0.50,0.625000,0.556250,0.285714,0.166667,0.142857,0.200000,0.333333,0.377778


# LSTM model for all datasets

In [12]:
import gc
import tensorflow as tf
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
import random
import os

# Reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

# Define evaluation metrics function
def evaluate_metrics(y_true, y_pred, y_pred_prob):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_pred_prob)
    }
    return metrics

# Define a function to process a single dataset and evaluate the model
def process_dataset(file_path):
    # Load dataset
    df = pd.read_csv(file_path)
    X = df.drop(columns=['Label', 'Subject_ID'])
    y = df['Label'].values

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Optional PCA
    use_pca = True
    n_components = 10
    if use_pca:
        pca = PCA(n_components=n_components)
        X_scaled = pca.fit_transform(X_scaled)

    # Handle imbalanced data (optional)
    smote = SMOTE(random_state=seed)
    X_balanced, y_balanced = smote.fit_resample(X_scaled, y)

    # Reshape for LSTM
    X_lstm = X_balanced.reshape((X_balanced.shape[0], 1, X_balanced.shape[1]))

    # K-Fold Cross-Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    cv_accuracy = []
    cv_auc = []

    for train_idx, val_idx in kf.split(X_lstm):
        X_train_fold, X_val_fold = X_lstm[train_idx], X_lstm[val_idx]
        y_train_fold, y_val_fold = y_balanced[train_idx], y_balanced[val_idx]

        # Build the model
        model = Sequential([
            Input(shape=(1, X_train_fold.shape[2])),  # Use Input layer
            LSTM(64, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy', metrics=['accuracy'])

        # Train the model with early stopping
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold),
                  epochs=20, batch_size=32, callbacks=[early_stopping], verbose=0)

        # Evaluate the fold
        y_val_pred_prob = model.predict(X_val_fold).ravel()
        y_val_pred = (y_val_pred_prob > 0.5).astype(int)
        cv_accuracy.append(accuracy_score(y_val_fold, y_val_pred))
        cv_auc.append(roc_auc_score(y_val_fold, y_val_pred_prob))

    # Final Model Training
    X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_balanced, test_size=0.2, random_state=seed)
    model = Sequential([
        Input(shape=(1, X_train.shape[2])),  # Use Input layer
        LSTM(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    model_checkpoint = ModelCheckpoint('best_model.keras', save_best_only=True)

    model.fit(X_train, y_train, validation_split=0.1, epochs=50, batch_size=32,
              callbacks=[early_stopping, lr_scheduler, model_checkpoint])

    # Evaluate on the test set
    y_test_pred_prob = model.predict(X_test).ravel()
    y_test_pred = (y_test_pred_prob > 0.5).astype(int)
    test_metrics = evaluate_metrics(y_test, y_test_pred, y_test_pred_prob)

    # Compile results
    results = {
        "Dataset": os.path.basename(file_path),
        "CV_Accuracy": np.mean(cv_accuracy),
        "CV_AUC": np.mean(cv_auc),
        "Test_Accuracy": test_metrics["accuracy"],
        "Test_Precision": test_metrics["precision"],
        "Test_Recall": test_metrics["recall"],
        "Test_F1_Score": test_metrics["f1_score"],
        "Test_AUC": test_metrics["auc"]
    }
    return results

# Process multiple datasets
def process_datasets(dataset_paths):
    all_results = []
    for dataset_path in dataset_paths:
        print(f"Processing dataset: {dataset_path}")
        result = process_dataset(dataset_path)
        all_results.append(result)
    return pd.DataFrame(all_results)

# Example usage
dataset_paths = [
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\putamen_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\all_no_put_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\table_only_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\left_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\right_hemisphere_mri_data_filtered.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\cleaned_mri_data_subjects_removed.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_1_functional_5.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_2_functional_5.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_1.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_2.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_3.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_4.csv",
        r"D:\\image_group_data\\team44\\output_for_freesurfer_table\\output\\combined\\combined_structural_3_functional_5.csv",
]

# Process all datasets and save results
final_results = process_datasets(dataset_paths)
final_results.to_csv(r"D:\image_group_data\team44\output_for_freesurfer_table\result\LSTM_all_ttest_bayesopt_smote.csv", index=False)

print(final_results)


Processing dataset: D:\\image_group_data\\team44\\CONN_Team44\\all_first_levlels\\optimize_output\\putamen_subjects_removed.csv
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 312ms/step - accuracy: 0.4487 - loss: 0.7158 - val_accuracy: 0.6364 - val_loss: 0.6943 - learning_rate: 0.0010
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.5402 - loss: 0.7040 - val_accuracy: 0.6364 - val_loss: 0.6888 - learning_rate: 0.0010
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.5652 - loss: 0.6908 - val_accuracy: 0.6364 - val_loss: 0.6837 - learning_rate: 0.0010
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.5826 - loss: 0.6831 - val_accuracy: 0.7273 - val_loss: 0.6790 - learning_rate: 0.0010
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.6584 - loss: 0.67

In [13]:
final_results

,Dataset,CV_Accuracy,CV_AUC,Test_Accuracy,Test_Precision,Test_Recall,Test_F1_Score,Test_AUC
0,putamen_subjects_removed.csv,0.632923,0.713738,0.461538,0.500000,0.642857,0.562500,0.666667
1,all_no_put_subjects_removed.csv,0.609846,0.679429,0.538462,0.625000,0.357143,0.454545,0.595238
2,table_only_subjects_removed.csv,0.571077,0.644540,0.692308,0.650000,0.928571,0.764706,0.744048
3,left_hemisphere_mri_data_filtered.csv,0.800923,0.920947,0.846154,0.750000,1.000000,0.857143,0.940476
4,right_hemisphere_mri_data_filtered.csv,0.842154,0.938763,0.846154,0.750000,1.000000,0.857143,0.916667
5,cleaned_mri_data_subjects_removed.csv,0.920308,0.992274,1.000000,1.000000,1.000000,1.000000,1.000000
6,combined_structural_1_functional_1.csv,0.800923,0.907483,0.807692,0.705882,1.000000,0.827586,0.982143
7,combined_structural_1_functional_2.csv,0.832615,0.929998,0.884615,0.800000,1.000000,0.888889,0.970238
8,combined_structural_1_functional_3.csv,0.880615,0.937789,0.846154,0.785714,0.916667,0.846154,0.946429
9,combined_structural_1_functional_4.csv,0.757353,0.873472,0.823529,0.875000,0.777778,0.823529,0.819444
